# Capitolo 5 (parte 2) — Transfer learning con ResNet18
Prima eseguire `python scarica_foto.py` (o mettere le proprie foto in `foto/{train,val,test}/<classe>/`).

In [1]:
import sys; sys.path.insert(0, "..")
from utils import fissa_seme
import numpy as np
import matplotlib.pyplot as plt

import torch, time, json
from torch import nn
from torchvision import datasets, models
from torch.utils.data import DataLoader
from PIL import Image
fissa_seme(42)

## Il dataset in cartelle

In [2]:
pesi = models.ResNet18_Weights.DEFAULT
prep = pesi.transforms()
print(prep)
ds = {s: datasets.ImageFolder(f"foto/{s}", transform=prep) for s in ("train", "val", "test")}
dl = {s: DataLoader(d, batch_size=32, shuffle=(s == "train")) for s, d in ds.items()}
classi = ds["train"].classes
print(classi, {s: len(d) for s, d in ds.items()})

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)
['cane', 'gatto'] {'train': 1000, 'val': 200, 'test': 400}


## Congelare il corpo, sostituire la testa

In [3]:
modello = models.resnet18(weights=pesi)
print(modello.fc)
for p in modello.parameters():
    p.requires_grad = False
modello.fc = nn.Linear(512, len(classi))
print("Addestrabili:", sum(p.numel() for p in modello.parameters() if p.requires_grad),
      "su", sum(p.numel() for p in modello.parameters()))

Linear(in_features=512, out_features=1000, bias=True)
Addestrabili: 1026 su 11177538


## Estrarre le caratteristiche una volta sola

In [4]:
modello.fc = nn.Identity()
modello.eval()
caratt = {}
t0 = time.time()
with torch.no_grad():
    for s in ("train", "val", "test"):
        xs, ys = [], []
        for xb, yb in dl[s]:
            xs.append(modello(xb)); ys.append(yb)
        caratt[s] = (torch.cat(xs), torch.cat(ys))
        print(s, caratt[s][0].shape, f"{time.time()-t0:.0f}s")

train torch.Size([1000, 512]) 53s


val torch.Size([200, 512]) 63s


test torch.Size([400, 512]) 84s


In [5]:
fissa_seme(42)
testa = nn.Linear(512, len(classi))
perdita_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(testa.parameters(), lr=1e-3)
X_tr, y_tr = caratt["train"]; X_va, y_va = caratt["val"]; X_te, y_te = caratt["test"]

def accuratezza(X, y):
    with torch.no_grad():
        return (testa(X).argmax(1) == y).float().mean().item()

migliore = (0, None, 0)
for epoca in range(30):
    perm = torch.randperm(len(X_tr))
    for i in range(0, len(X_tr), 32):
        idx = perm[i:i+32]
        opt.zero_grad()
        perdita_fn(testa(X_tr[idx]), y_tr[idx]).backward()
        opt.step()
    acc_val = accuratezza(X_va, y_va)
    if acc_val > migliore[0]:
        migliore = (acc_val, {k: v.clone() for k, v in testa.state_dict().items()}, epoca + 1)
    if epoca % 5 == 4 or epoca == 0:
        print(f"Epoca {epoca+1:2d}: acc train {accuratezza(X_tr, y_tr):.2%} | val {acc_val:.2%}")
testa.load_state_dict(migliore[1])
print(f"Migliore: epoca {migliore[2]} (val {migliore[0]:.2%})   Test: {accuratezza(X_te, y_te):.2%}")

Epoca  1: acc train 95.90% | val 95.50%
Epoca  5: acc train 98.60% | val 97.00%
Epoca 10: acc train 99.40% | val 97.50%
Epoca 15: acc train 99.90% | val 97.00%


Epoca 20: acc train 99.90% | val 97.00%


Epoca 25: acc train 100.00% | val 97.50%
Epoca 30: acc train 100.00% | val 96.00%
Migliore: epoca 6 (val 97.50%)   Test: 99.25%


## Rimettere la testa sul corpo, salvare (per il capitolo 7)

In [6]:
modello.fc = testa
modello.eval()
torch.save(modello.state_dict(), "../cap07_inferenza/gatti_cani.pt")
json.dump({"classi": classi, "architettura": "resnet18", "torch": torch.__version__},
          open("../cap07_inferenza/gatti_cani.json", "w"))

## Una foto propria

In [7]:
percorso = "foto/test/gatto/0600.jpg"      # sostituire con una propria foto
img = prep(Image.open(percorso).convert("RGB")).unsqueeze(0)
with torch.no_grad():
    prob = torch.softmax(modello(img), dim=1)[0]
for classe, p in zip(classi, prob):
    print(f"{classe}: {p:.1%}")

cane: 0.0%
gatto: 100.0%


## Fine-tuning (opzionale, lento su CPU): scongelare gli ultimi strati

In [8]:
# for p in modello.layer4.parameters():
#     p.requires_grad = True
# opt = torch.optim.Adam([p for p in modello.parameters() if p.requires_grad], lr=1e-4)
# for epoca in range(3):
#     modello.train()
#     for xb, yb in dl["train"]:
#         opt.zero_grad(); perdita_fn(modello(xb), yb).backward(); opt.step()
#     modello.eval(); corretti = 0
#     with torch.no_grad():
#         for xb, yb in dl["val"]:
#             corretti += (modello(xb).argmax(1) == yb).sum().item()
#     print(f"Epoca {epoca+1}: val {corretti/len(ds['val']):.2%}")